# REDES NEURAIS ARTIFICIAIS

# PARTE 1: Classificação Binária - Diagnóstico de Câncer de Mama

Base de dados: Breast Cancer Wisconsin (Diagnostic) -
https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

In [ ]:
# após instalar as bibliotecas, é necessário reiniciar o notebook
# Para isso, rode esta célula sozinha antes de prosseguir.
exit()

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import sklearn
import scikeras
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras import backend as k

In [ ]:
pd.__version__, tf.__version__, sklearn.__version__, scikeras.__version__

### Carregar os dados

**1) Carregue a base de dados, faça a divisão de treino e teste (para isso, utilize a função train_test_split do sklearn), como o tamanho da base de teste de 0.25.**

In [ ]:
X = pd.read_csv('dados_breast.csv')
y = pd.read_csv('rotulos_breast.csv')

X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=0
)

In [ ]:
X_treinamento.shape, y_treinamento.shape

In [ ]:
X_teste.shape, y_teste.shape

### Estrutura da Rede Neural Artificial e Teste

In [ ]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from tensorflow.keras.models import Sequential
import tensorflow as tf
from tensorflow.keras import backend as k

**2) Crie a RNA com as seguintes configurações:**

**a) uma camada de entrada com 30 neurônios;**

**b) uma camada oculta densa com 16 neurônios. Porque a camada oculta tem essa quantidade de neurônios?**

A camada oculta com 16 neurônios é uma escolha heurística comum: utiliza-se um número entre o tamanho da entrada (30) e da saída (1), geralmente uma potência de 2 para eficiência computacional. 16 é suficiente para capturar padrões não-lineares do problema sem tornar o modelo excessivamente complexo.

**c) adicione na camada oculta a função de ativação relu e inicialize os pesos com Random uniform initializer;**

**d) adicione a camada de saída com a função de ativação sigmoid. Porque foi utilizada esta função para a saída da RNA?**

A função sigmoid é utilizada na saída porque produz valores entre 0 e 1, que podem ser interpretados como probabilidades. Como o problema é de classificação binária (maligno/benigno), precisamos de uma saída que represente a probabilidade de pertencer à classe 1.

**3) Porque utilizamos a classe chamada Sequential para a RNA?**

A classe Sequential é utilizada porque permite construir a rede camada por camada de forma linear (sequencial), onde a saída de uma camada é a entrada da próxima. É a forma mais simples e direta de criar uma RNA com fluxo unidirecional, adequada para a maioria dos problemas de classificação.

In [ ]:
k.clear_session()
rede_neural = Sequential([
    tf.keras.layers.InputLayer(shape=(30,)),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dense(units=1, activation='sigmoid')
])

**4) A partir da RNA gerada, explique o que são os valores apresentados na tabela da rede_neural.summary()**

- **Layer (type)**: Nome e tipo de cada camada da rede.
- **Output Shape**: Dimensão da saída de cada camada. `(None, 16)` significa que para cada amostra (None = qualquer quantidade de amostras) há 16 saídas.
- **Param #**: Número de parâmetros treináveis. Para a camada densa de 16 neurônios: (30 entradas × 16 neurônios) + 16 biases = 496 parâmetros. Para a camada de saída: (16 × 1) + 1 bias = 17 parâmetros. Total: 513 parâmetros.
- **Trainable params**: parâmetros que serão atualizados durante o treinamento.
- **Non-trainable params**: parâmetros fixos (ex.: em camadas congeladas).

In [ ]:
rede_neural.summary()

**5) Adicione um otimizador Adam e especifique a classe loss binário - binary crossentropy e a métrica de acurácia binária.**

**6) Para que servem os otimizadores? Como o otimizador Adam funciona?**

Os otimizadores servem para atualizar os pesos da rede neural de forma a minimizar a função de perda (loss). O Adam (Adaptive Moment Estimation) combina duas técnicas:
- **Momentum**: acumula gradientes passados para suavizar a atualização (evita oscilações);
- **RMSProp**: adapta a taxa de aprendizado para cada parâmetro individualmente com base na magnitude dos gradientes recentes.
O resultado é um otimizador eficiente, robusto e que converge rapidamente, sendo uma das escolhas padrão na prática.

**7) Depois de estruturados os parâmetros da RNA, utilize a função .fit para fazer o treinamento da rede. Com batch_size=10, quantos batches serão utilizados ao total?**

O conjunto de treinamento tem 426 amostras (75% de 569). Com batch_size=10: ⌈426 / 10⌉ = 43 batches por época × 100 épocas = 4.300 atualizações de pesos no total.

**8) Defina o número de épocas em que ocorre o treinamento igual a 100.**

In [ ]:
rede_neural.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy']
)

In [ ]:
rede_neural.fit(X_treinamento, y_treinamento, batch_size=10, epochs=100)

**9) Crie uma variável chamada previsoes para realizar a previsão dos dados de teste (X_teste). O resultado da rede deve ser um valor entre 0 e 1. Porque isso acontece?**

Isso acontece porque a camada de saída utiliza a função sigmoid, que mapeia qualquer valor real para o intervalo (0, 1). Esse valor representa a probabilidade da amostra pertencer à classe positiva (tumor maligno).

In [ ]:
previsoes = rede_neural.predict(X_teste)

**10) Converta o resultado que está em probabilidade para valores binários com um limiar (threshold) de 0.5.**

In [ ]:
previsoes = previsoes > 0.5

In [ ]:
previsoes

**9) Explique o resultado da RNA.**

Após a conversão com threshold 0.5, cada previsão é True (classe 1 = maligno) ou False (classe 0 = benigno). A função `evaluate` retorna o valor da loss (binary_crossentropy) e a acurácia binária no conjunto de teste, mostrando o desempenho do modelo em dados não vistos durante o treinamento.

In [ ]:
rede_neural.evaluate(X_teste, y_teste)

## Camadas e Otimização da RNA

**10) Adicione mais uma camada oculta densa com 16 neurônios com a função de ativação relu e inicialize os pesos com Random uniform initializer. Qual o total de parâmetros da RNA agora?**

Com a segunda camada oculta:
- Camada 1: (30×16) + 16 = 496 parâmetros
- Camada 2: (16×16) + 16 = 272 parâmetros
- Saída: (16×1) + 1 = 17 parâmetros
- **Total: 785 parâmetros**

In [ ]:
k.clear_session()
rede_neural = Sequential([
    tf.keras.layers.InputLayer(shape=(30,)),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dense(units=1, activation='sigmoid')
])

In [ ]:
rede_neural.summary()

**11) Abaixo são adicionados os parâmetros do otimizador: learning_rate e clipvalue. O que eles fazem?**

- **learning_rate (0.001)**: controla o tamanho do passo dado na descida do gradiente. Valores muito altos fazem o modelo divergir; valores muito baixos tornam o treinamento lento. 0.001 é o valor padrão do Adam.
- **clipvalue (0.5)**: limita o valor máximo dos gradientes durante o backpropagation. Evita o problema de "gradiente explosivo", onde gradientes muito grandes desestabilizam o treinamento.

In [ ]:
otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
rede_neural.compile(optimizer=otimizador, loss='binary_crossentropy', metrics=['binary_accuracy'])

**12) Teste novamente a RNA. Aumentar a quantidade de camadas melhorou ou piorou os resultados? Explique o que aconteceu com a RNA e porque.**

Com mais uma camada oculta, a rede ganha mais capacidade de aprendizado, podendo capturar padrões mais complexos. Em geral, o desempenho no treino melhora, porém pode ocorrer **overfitting**: o modelo aprende os dados de treino com excessiva precisão mas generaliza pior para dados novos. O resultado no conjunto de teste pode melhorar levemente ou se manter estável; se piorar, é sinal de overfitting.

In [ ]:
rede_neural.fit(X_treinamento, y_treinamento, batch_size=10, epochs=100)

In [ ]:
previsoes = rede_neural.predict(X_teste)

In [ ]:
rede_neural.evaluate(X_teste, y_teste)

## K-Fold Cross Validation

**13) Explique como esta rede foi configurada e como é possível chegar no resultado dela. O que é necessário fazer?**

A validação cruzada K-Fold divide o conjunto de dados em K partes (folds) iguais. Em cada iteração, K-1 folds são usados para treino e 1 fold para validação, rotacionando-se qual fold é o de validação. Ao final, obtêm-se K métricas de acurácia, cujas média e desvio padrão representam o desempenho esperado e a variabilidade do modelo. Aqui utilizamos K=10 folds, o que significa 10 treinos completos da rede. A RNA é encapsulada com `KerasClassifier` para funcionar com a API do scikit-learn.

**14) Calcule também o Desvio Padrão dos resultados para avaliar o modelo. O que é possível concluir com esse resultado?**

Um desvio padrão baixo indica que o modelo tem desempenho consistente entre os folds, ou seja, generaliza bem. Um desvio padrão alto sugere que o modelo é instável e sensível a quais dados são usados para treino, podendo indicar overfitting ou poucos dados.

In [ ]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from tensorflow.keras.models import Sequential
from tensorflow.keras import backend as k

In [ ]:
def criar_rede():
    k.clear_session()
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
        tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
        tf.keras.layers.Dense(units=1, activation='sigmoid')
    ])
    otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
    rede_neural.compile(optimizer=otimizador, loss='binary_crossentropy', metrics=['binary_accuracy'])
    return rede_neural

In [ ]:
rede_neural = KerasClassifier(model=criar_rede, epochs=100, batch_size=10)

In [ ]:
resultados = cross_val_score(estimator=rede_neural, X=X, y=y, cv=10, scoring='accuracy')

In [ ]:
resultados

In [ ]:
print(f'Média de acurácia: {resultados.mean():.4f}')
print(f'Desvio padrão:     {resultados.std():.4f}')

## Overfitting e Dropout

**14) Como vimos, o modelo anterior sofreu overfitting. Para reduzir esse problema, aplicamos a técnica de regularização conhecida como dropout. Aplique o dropout de 20% na primeira e segunda camada oculta. O que acontece com os resultados? E o Desvio Padrão?**

O Dropout desativa aleatoriamente 20% dos neurônios a cada passo de treinamento. Isso impede que a rede dependa demais de neurônios específicos, forçando o aprendizado de representações mais robustas e distribuídas. Espera-se que a acurácia média se mantenha boa e que o desvio padrão reduza, indicando menor variabilidade e melhor generalização.

In [ ]:
def criar_rede_dropout():
    k.clear_session()
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=1, activation='sigmoid')
    ])
    otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
    rede_neural.compile(optimizer=otimizador, loss='binary_crossentropy', metrics=['binary_accuracy'])
    return rede_neural

In [ ]:
rede_neural = KerasClassifier(model=criar_rede_dropout, epochs=100, batch_size=10)

In [ ]:
resultados = cross_val_score(estimator=rede_neural, X=X, y=y, cv=10, scoring='accuracy')

In [ ]:
resultados

In [ ]:
print(f'Média de acurácia (com Dropout): {resultados.mean():.4f}')
print(f'Desvio padrão (com Dropout):     {resultados.std():.4f}')

## Tuning dos Hiperparâmetros (técnicas de ajuste)

Hyperparameter Tuning (Ajuste de Hiperparâmetros) é o processo geral de encontrar os melhores valores para os hiperparâmetros de um modelo de machine learning.
Exemplos de hiperparâmetros:

- Taxa de aprendizado (learning rate)
- Número de árvores em um RandomForest
- Número de camadas ou neurônios em uma rede neural
- Tipo de kernel em um SVM

O tuning pode ser feito por diferentes técnicas, como:

- Grid Search
- Random Search
- Bayesian Optimization, etc.

**15) Descreva como a RNA foi configurada para fazer o processo de tuning.**

A função `criar_rede` foi parametrizada para aceitar os hiperparâmetros como argumentos (`optimizer`, `loss`, `kernel_initializer`, `activation`, `neurons`). O `KerasClassifier` encapsula essa função e o `GridSearchCV` testa todas as combinações definidas em `parametros`, avaliando cada configuração com validação cruzada de 5 folds. Ao final, `best_params_` e `best_score_` retornam a combinação de hiperparâmetros que maximizou a acurácia média.

In [ ]:
def criar_rede(optimizer, loss, kernel_initializer, activation, neurons):
    k.clear_session()
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=neurons, activation=activation, kernel_initializer=kernel_initializer),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=neurons, activation=activation, kernel_initializer=kernel_initializer),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=1, activation='sigmoid')
    ])
    rede_neural.compile(optimizer=optimizer, loss=loss, metrics=['binary_accuracy'])
    return rede_neural

In [ ]:
rede_neural = KerasClassifier(model=criar_rede)

In [ ]:
parametros = {
    'batch_size': [10, 30],
    'epochs': [50, 100],
    'model__optimizer': ['adam', 'sgd'],
    'model__loss': ['binary_crossentropy', 'hinge'],
    'model__kernel_initializer': ['random_uniform', 'normal'],
    'model__activation': ['relu', 'tanh'],
    'model__neurons': [16, 8]
}

In [ ]:
# caso tiver problema com a configuração anterior, testar essa
parametros = {
    'batch_size': [10, 30],
    'epochs': [50],
    'model__optimizer': ['adam'],
    'model__loss': ['binary_crossentropy'],
    'model__kernel_initializer': ['random_uniform', 'normal'],
    'model__activation': ['relu'],
    'model__neurons': [16]
}

In [ ]:
parametros

In [ ]:
grid_search = GridSearchCV(estimator=rede_neural, param_grid=parametros,
                           scoring='accuracy', cv=5)

In [ ]:
grid_search = grid_search.fit(X, y)

In [ ]:
print(grid_search)

In [ ]:
melhores_parametros = grid_search.best_params_
print(melhores_parametros)

In [ ]:
melhor_precisao = grid_search.best_score_
print(melhor_precisao)

**16) É possível melhorar ainda mais a acurácia da RNA? Como?**

Sim. Algumas estratégias possíveis:
- **Aumentar o espaço de busca do Grid Search**: testar mais combinações de neurônios, camadas, otimizadores e taxas de aprendizado.
- **Normalização dos dados**: aplicar StandardScaler ou MinMaxScaler nas features pode acelerar a convergência e melhorar a acurácia.
- **Aumentar épocas**: mais épocas permitem que a rede aprenda melhor, desde que combinado com dropout para evitar overfitting.
- **Adicionar mais camadas**: arquiteturas mais profundas capturam padrões mais complexos.
- **Batch Normalization**: normaliza as ativações entre camadas, estabilizando e acelerando o treinamento.
- **Early Stopping**: parar o treinamento quando a loss de validação para de melhorar, evitando overfitting.

# Para salvar o seu modelo de RNA

In [ ]:
# Treina o modelo final com os melhores parâmetros antes de salvar
k.clear_session()
classificador = Sequential([
    tf.keras.layers.InputLayer(shape=(30,)),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(units=1, activation='sigmoid')
])
otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
classificador.compile(optimizer=otimizador, loss='binary_crossentropy', metrics=['binary_accuracy'])
classificador.fit(X_treinamento, y_treinamento, batch_size=10, epochs=100)

In [ ]:
classificador.save('classificador_breast.keras')

In [ ]:
classificador_novo = tf.keras.models.load_model('classificador_breast.keras')

In [ ]:
classificador_novo.summary()